# Optional: score inside Fabric with PREDICT (option A)

**The default is option B**: `score.yml` runs `jobs/batch-score.yml` in Azure ML every night.
Use this notebook only if your team prefers to score inside Fabric, next to the data.

What it does: reads a lakehouse table, scores it with the approved model using
Fabric's `PREDICT` (`MLFlowTransformer`), and writes a predictions table that Power BI can read.

Before you start:
1. The **Azure ML registry stays the source of truth**: only score a version that is
   `status=champion` there.
2. That version must be **copied to the Fabric model registry** under the same name
   and version (see the last cell).
3. Run this notebook in a Fabric workspace attached to the lakehouse with the input table.

Limits of PREDICT: the model needs an MLflow signature (ours has one), and it returns
the output of `predict()`, so here you get `predicted_failure` (0/1) but **not** the
failure probability that option B writes.

## Settings

In [ ]:
# ASSUMPTION: names of the model in the Fabric registry and of the lakehouse
# tables. Change them to yours.
MODEL_NAME = "tool-wear"
MODEL_VERSION = 1  # the champion version from the Azure ML registry
INPUT_TABLE = "tool_wear_to_score"  # columns already in snake_case, as in training
OUTPUT_TABLE = "ml_tool_wear_predictions"  # Gold table, = predictions_table in config.yml

## Score with PREDICT

In [ ]:
import mlflow
from pyspark.sql import functions as F
from synapse.ml.predict import MLFlowTransformer

model_uri = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
# The input columns the model was trained on, from its MLflow signature.
features = mlflow.models.get_model_info(model_uri).signature.inputs.input_names()

df = spark.read.table(INPUT_TABLE)
missing = set(features) - set(df.columns)
assert not missing, f"Input table is missing columns: {missing}"

predict = MLFlowTransformer(
    inputCols=features,
    outputCol="predicted_failure",
    modelName=MODEL_NAME,
    modelVersion=MODEL_VERSION,
)
predictions = (
    predict.transform(df)
    .withColumn("model_uri", F.lit(model_uri))
    .withColumn("scored_at", F.current_timestamp())
)
predictions.write.mode("overwrite").format("delta").saveAsTable(OUTPUT_TABLE)
print(f"Wrote {predictions.count()} rows to {OUTPUT_TABLE}")

## Copy the approved model from Azure ML to Fabric (once per approved version)

A sketch, not tested: how Azure ML and Fabric authenticate to each other is still an
open question in the design (section 8). Steps:

1. Download the champion version from the Azure ML registry with MLflow
   (`mlflow.artifacts.download_artifacts("models:/tool-wear/<version>")`, with
   `MLFLOW_TRACKING_URI` pointing to the Azure ML workspace).
2. In this Fabric notebook, register the downloaded folder in the Fabric registry:
   `mlflow.register_model("file:///lakehouse/default/Files/models/tool-wear/<version>", "tool-wear")`.
3. Check that the Fabric version number and the Azure ML version match, or note both
   in `model-card.md`.

`# ASSUMPTION:` the platform team decides how this copy is automated and who may do it.